# Training Job 제출 (한 번 실행하고 자면 됨)

**셀 2개만 실행:**
1. U-Net 제출 (~4-5시간)
2. DenseNet 제출 (~8-9시간)

둘 다 제출하고 자면 됨. Training Job이 알아서:
- CheXmask/CSV 다운로드
- 마스크 디코딩 / PA CSV 생성
- 이미지 S3 선택 다운로드
- 학습 + 모델 저장

**예상 비용: ~$4-7 (스팟)**

In [ ]:
# ======================================
# U-Net 세그멘테이션 Training Job 제출
# ======================================
import boto3, json, os, tarfile, time

sm = boto3.client('sagemaker', region_name='ap-northeast-2')
s3 = boto3.client('s3')

BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
IMAGE_BUCKET = 'say1-pre-project-5'
ROLE = 'arn:aws:iam::666803869796:role/SKKU_SageMaker_Role'
CONTAINER = '763104351884.dkr.ecr.ap-northeast-2.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker'

# train_unet.py 패키징 + 업로드
UNET_SCRIPT = None
for p in [
    '/home/ec2-user/SageMaker/forpreproject/layer1_segmentation/train_unet.py',
    '/home/ec2-user/SageMaker/train_unet.py',
]:
    if os.path.exists(p):
        UNET_SCRIPT = p; break

if UNET_SCRIPT is None:
    UNET_SCRIPT = '/tmp/train_unet.py'
    s3.download_file(BUCKET, 'code/train_unet.py', UNET_SCRIPT)

tar_path = '/tmp/unet_sourcedir.tar.gz'
with tarfile.open(tar_path, 'w:gz') as tar:
    tar.add(UNET_SCRIPT, arcname='train_unet.py')
s3.upload_file(tar_path, BUCKET, 'code/unet_sourcedir.tar.gz')
print(f'train_unet.py 패키징 완료')

# Training Job 제출
JOB_NAME = 'unet-lung-heart-v3'

response = sm.create_training_job(
    TrainingJobName=JOB_NAME,
    RoleArn=ROLE,
    AlgorithmSpecification={
        'TrainingImage': CONTAINER,
        'TrainingInputMode': 'File'
    },
    HyperParameters={
        'sagemaker_program': 'train_unet.py',
        'sagemaker_submit_directory': f's3://{BUCKET}/code/unet_sourcedir.tar.gz',
        'sagemaker_region': 'ap-northeast-2',
        'epochs': '50',
        'batch-size': '8',
        'lr': '0.0001',
        'image-size': '512',
        'num-workers': '4',
        'image-bucket': IMAGE_BUCKET,
        'work-bucket': BUCKET,
    },
    ResourceConfig={
        'InstanceType': 'ml.g5.xlarge',
        'InstanceCount': 1,
        'VolumeSizeInGB': 150
    },
    CheckpointConfig={
        'S3Uri': f's3://{BUCKET}/checkpoints/unet-lung-heart/'
    },
    OutputDataConfig={
        'S3OutputPath': f's3://{BUCKET}/output/'
    },
    EnableManagedSpotTraining=True,
    StoppingCondition={
        'MaxRuntimeInSeconds': 28800,
        'MaxWaitTimeInSeconds': 172800
    },
    Tags=[{'Key': 'name', 'Value': 'say2-preproject-6team-hyunwoo'}]
)

print('=' * 60)
print(f'U-Net Training Job 제출 완료!')
print(f'Job: {JOB_NAME}')
print(f'Phase 1: CheXmask 다운로드 + 이미지 다운로드 (~40분)')
print(f'Phase 2: U-Net 학습 50에폭 (~3시간)')
print(f'예상 총 소요: ~4시간 | 비용: ~$2-3 (스팟)')
print('=' * 60)

In [ ]:
# ======================================
# DenseNet-121 전체 PA Training Job v3 제출
# v2 대비 변경: 볼륨 150GB, g5.xlarge, 체크포인트 격리, 디스크 안전장치 추가
# ======================================
import boto3, json, os, tarfile

sm = boto3.client('sagemaker', region_name='ap-northeast-2')
s3 = boto3.client('s3')

BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'
IMAGE_BUCKET = 'say1-pre-project-5'
ROLE = 'arn:aws:iam::666803869796:role/SKKU_SageMaker_Role'
CONTAINER = '763104351884.dkr.ecr.ap-northeast-2.amazonaws.com/pytorch-training:2.8.0-gpu-py312-cu129-ubuntu22.04-sagemaker'

# sourcedir는 이미 S3에 업로드됨 (로컬에서 패키징 완료)
# 확인만 하고 없으면 로컬에서 다시 패키징
try:
    s3.head_object(Bucket=BUCKET, Key='code/densenet_full_sourcedir.tar.gz')
    print('✅ sourcedir 확인: s3://...code/densenet_full_sourcedir.tar.gz')
except:
    print('⚠️ sourcedir 없음 — 로컬에서 패키징 시도')
    DN_SCRIPT = None
    for p in [
        '/home/ec2-user/SageMaker/forpreproject/layer2_detection/densenet/train.py',
        '/home/ec2-user/SageMaker/train_densenet.py',
    ]:
        if os.path.exists(p):
            DN_SCRIPT = p; break
    if DN_SCRIPT is None:
        DN_SCRIPT = '/tmp/train_densenet.py'
        s3.download_file(BUCKET, 'code/densenet/train.py', DN_SCRIPT)
    tar_path = '/tmp/densenet_full_sourcedir.tar.gz'
    with tarfile.open(tar_path, 'w:gz') as tar:
        tar.add(DN_SCRIPT, arcname='train.py')
    s3.upload_file(tar_path, BUCKET, 'code/densenet_full_sourcedir.tar.gz')
    print('✅ train.py 패키징 + 업로드 완료')

# Training Job 제출 — v3
JOB_NAME = 'densenet121-full-pa-v3'

response = sm.create_training_job(
    TrainingJobName=JOB_NAME,
    RoleArn=ROLE,
    AlgorithmSpecification={
        'TrainingImage': CONTAINER,
        'TrainingInputMode': 'File'
    },
    HyperParameters={
        'sagemaker_program': 'train.py',
        'sagemaker_submit_directory': f's3://{BUCKET}/code/densenet_full_sourcedir.tar.gz',
        'sagemaker_region': 'ap-northeast-2',
        'batch-size': '32',
        'stage1-epochs': '5',
        'stage2-epochs': '25',
        'image-bucket': IMAGE_BUCKET,
        'work-bucket': BUCKET,
    },
    InputDataConfig=[
        {
            'ChannelName': 'csv',
            'DataSource': {
                'S3DataSource': {
                    'S3DataType': 'S3Prefix',
                    'S3Uri': f's3://{BUCKET}/mimic-cxr-csv/',
                    'S3DataDistributionType': 'FullyReplicated'
                }
            }
        }
    ],
    ResourceConfig={
        'InstanceType': 'ml.g5.xlarge',   # v2: g4dn → v3: g5 (더 빠른 GPU)
        'InstanceCount': 1,
        'VolumeSizeInGB': 150              # v2: 100GB → v3: 150GB (디스크 부족 방지)
    },
    CheckpointConfig={
        'S3Uri': f's3://{BUCKET}/checkpoints/densenet121-full-pa-v3/'  # v2와 격리
    },
    OutputDataConfig={
        'S3OutputPath': f's3://{BUCKET}/output/'
    },
    EnableManagedSpotTraining=True,
    StoppingCondition={
        'MaxRuntimeInSeconds': 21600,      # 6시간
        'MaxWaitTimeInSeconds': 172800     # 스팟 대기 최대 48시간
    },
    Tags=[{'Key': 'name', 'Value': 'say2-preproject-6team-hyunwoo'}]
)

print('=' * 60)
print(f'🚀 DenseNet v3 Training Job 제출 완료!')
print(f'Job: {JOB_NAME}')
print(f'인스턴스: ml.g5.xlarge (24GB GPU, 150GB 볼륨)')
print(f'v2 대비 개선: 볼륨 +50GB, 디스크 모니터링, atomic 저장')
print(f'Phase 1: CSV + PA 이미지 다운로드 (~15분)')
print(f'Phase 2: Stage1 5에폭 + Stage2 25에폭 (~5-6시간)')
print(f'예상 비용: ~$2-3 (스팟)')
print('=' * 60)

---
## (다음날) 상태 확인

In [ ]:
# 상태 확인
import boto3, json
sm = boto3.client('sagemaker', region_name='ap-northeast-2')
BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'

for job in ['densenet121-full-pa-v3']:
    try:
        desc = sm.describe_training_job(TrainingJobName=job)
        status = desc['TrainingJobStatus']
        detail = desc.get('SecondaryStatus', '')
        print(f'\n{job}')
        print(f'  Status: {status} ({detail})')
        if status == 'Completed':
            t = desc.get('TrainingTimeInSeconds', 0)
            b = desc.get('BillableTimeInSeconds', 0)
            print(f'  Training: {t/60:.0f}분 | Billable: {b/60:.0f}분')
            print(f'  Model: s3://{BUCKET}/output/{job}/output/model.tar.gz')
        elif status == 'Failed':
            print(f'  Error: {desc.get("FailureReason", "?")}')
        elif status == 'InProgress':
            print(f'  실행 중... 나중에 다시 확인')
    except Exception as e:
        print(f'\n{job}: {e}')

In [ ]:
# 완료된 Job 결과 다운로드
import tarfile, json
BUCKET = 'pre-project-practice-hyunwoo-666803869796-ap-northeast-2-an'

for job, model_name in [
    ('unet-lung-heart-v3', 'unet_lung_heart'),
    ('densenet121-full-pa-v2', 'densenet121_full_pa'),
]:
    try:
        tar_path = f'/tmp/{job}_model.tar.gz'
        !aws s3 cp s3://{BUCKET}/output/{job}/output/model.tar.gz {tar_path}
        extract_dir = f'/tmp/{job}_model'
        with tarfile.open(tar_path, 'r:gz') as tar:
            tar.extractall(extract_dir)
        
        with open(f'{extract_dir}/results.json') as f:
            r = json.load(f)
        
        print(f'\n{"="*50}')
        print(f'{job}')
        print(f'{"="*50}')
        for k, v in r.items():
            if isinstance(v, float):
                print(f'  {k}: {v:.4f}')
            elif isinstance(v, dict):
                for kk, vv in v.items():
                    if isinstance(vv, float):
                        print(f'    {kk}: {vv:.4f}')
        
        # Lambda 배포용 복사
        !aws s3 cp {extract_dir}/best_model.pth s3://{BUCKET}/models/{model_name}.pth
        print(f'  -> s3://{BUCKET}/models/{model_name}.pth')
    except Exception as e:
        print(f'\n{job}: {e}')